# Day 6 — Data structures: list, tuple, set, dict
Objectives:
- Understand properties and performance.
- Choose the right structure for the task.
- Common patterns: frequency count, dedupe, grouping.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-06`. Read
`python/ds-60day/companion-guides/day06_data_structures.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Python's core collections answer different questions. A list keeps
ordered, possibly repeated values and can change. A tuple is an ordered
fixed record-like sequence. A set represents unique hashable members
and makes membership tests fast. A dictionary maps unique hashable keys
to values.

“Mutable” means an object can change in place; “hashable” means it has a
stable hash suitable for a set member or dictionary key. Copying a
collection is also about reference depth: a shallow copy creates a new
outer container but still points to the same nested objects. Choose a
structure from the meaning of order, duplicates, lookup, and mutation,
not simply from familiar syntax.

### Vocabulary

- **sequence:** an ordered collection addressable by position.
- **mapping:** a collection that associates keys with values.
- **mutable:** able to change in place after creation.
- **hashable:** having a stable hash and equality behavior while stored.
- **membership:** the question whether a value is present.
- **shallow copy:** a new outer container that shares referenced inner objects.

## Syntax anatomy

`counts[key] = counts.get(key, 0) + amount` first asks the dictionary
for `key`, substitutes `0` only when it is absent, adds `amount`, and
writes the new total back. `value in seen` is a membership expression;
for a set or dictionary it is normally much cheaper than scanning a
long list.

### Worked example 1 — Preserve order while removing repeats

Use one structure for fast membership and another for ordered output. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
values = ["a", "b", "a", "c", "b"]
seen = set()
unique_in_order = []
for value in values:
    if value not in seen:
        seen.add(value)
        unique_in_order.append(value)
unique_in_order

**Expected observation:** `['a', 'b', 'c']`. Converting straight to a set would represent uniqueness but would not express first-seen order.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Aggregate pairs into a dictionary

Make the missing-key starting value explicit. Predict first; then run the next cell.

In [ ]:
purchases = [("tea", 2), ("coffee", 1), ("tea", 3)]
totals = {}
for product, quantity in purchases:
    totals[product] = totals.get(product, 0) + quantity
totals

**Expected observation:** `{'tea': 5, 'coffee': 1}`. Tuple unpacking gives names to the two fields in each pair.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. When an unhashable-type error appears, inspect the set member or dictionary key and ask whether its identity can be a tuple.
2. When nested data changes through a copy, determine whether the copy was shallow.
3. Do not depend on a set's display order; sort only when presentation needs a deterministic order.
4. Use `.get`, `setdefault`, `defaultdict`, or `Counter` according to the missing-key behavior you want.

**Alternative to compare:** `dict.fromkeys(values)` is a compact stable de-duplication technique for hashable values, while an explicit loop adapts to a derived key.

**Boundary to test:** Nested mutable values, unhashable records, missing keys, empty inputs, and repeated records with the same identity expose collection-policy choices.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
from collections import Counter, defaultdict
words = 'this is a test this is'.split()
Counter(words)

# Dedupe
list(set([1,2,2,3,3,3]))

# Grouping with defaultdict
groups = defaultdict(list)
pairs = [('a',1),('b',2),('a',3)]
for k,v in pairs:
    groups[k].append(v)
dict(groups)


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Implement `stable_unique(values)` for `['b', 'a', 'b', 'c', 'a']`.
   **Expected result:** `['b', 'a', 'c']`. **Constraints:** preserve the first occurrence, leave the input unchanged, use a set for membership and a list for output, and state that items must be hashable.
   **Verify:** test the example, an empty list, and a list with no duplicates.

2. Given `[('east', 4), ('west', 2), ('east', 3)]`, build three mappings: region to list of all values, region to set of unique values, and region to numeric total. **Expected totals:** `{'east': 7, 'west': 2}`. **Constraints:** make the missing-key initial value explicit and preserve encounter order in the list version.
   **Verify:** assert all three outputs.

### Additional mastery practice

Choose structures by semantics—ordering, uniqueness, lookup, and mutability—not by habit. State what duplicates and missing keys mean.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict the length and membership behavior of `{3, 1, 3, 2}`. Why must display order not be treated as a sorting guarantee?
   **Progressive hint:** Sets enforce uniqueness and optimize membership, not presentation order.
   **Verify:** Assert the set has length `3` and contains `1`, `2`, and `3`; compare values as a set rather than asserting display iteration order.
4. **Tracing:** Trace shallow copying for `original = [[1], [2]]; copied = original.copy(); copied[0].append(9)`.
   **Progressive hint:** The outer lists differ but still refer to the same inner lists.
   **Verify:** Assert `original is not copied` but `original[0] is copied[0]`, then confirm appending through the copy appears in both nested lists.
5. **Implementation:** Implement `invert_multimap(mapping)` so values become keys and each new key maps to a list of original keys in encounter order.
   **Progressive hint:** Use `setdefault` or `defaultdict(list)`.
   **Verify:** Assert exact output lists and encounter order for a mapping where two original keys share a value; also test an empty mapping.
6. **Debugging:** Repair code that attempts to use a list as a dictionary key. Explain hashability and choose a tuple when the sequence is an identity.
   **Progressive hint:** Dictionary keys must have a stable hash while stored.
   **Verify:** Show the list-key version raises `TypeError`, then assert the tuple-key replacement retrieves the intended value and remains unmodified.
7. **Edge case and explanation:** Extend stable de-duplication to records using a `key` function, then test repeated dictionaries whose IDs match but other fields differ.
   **Progressive hint:** Store hashable derived keys while returning original records.
   **Verify:** Use repeated record dictionaries with the same ID; assert the first complete record is returned once, input order is preserved, and the input remains unchanged.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Implement `stable_unique(values)` for `['b', 'a', 'b', 'c', 'a']`. **Expected result:** `['b', 'a', 'c']`. **Constraints:** preserve the first occurrence, leave the input unchanged, use a set for membership and a list for output, and state that items must be hashable. **Verify:** test the example, an empty list, and a list with no duplicates.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Implement `stable_unique(values)` for `['b', 'a', 'b', 'c', 'a']`. `['b', 'a', 'c']`. preserve the first occurrence, leave the input unchanged, use a set for membership and a li...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Given `[('east', 4), ('west', 2), ('east', 3)]`, build three mappings: region to list of all values, region to set of unique values, and region to numeric total. **Expected totals:** `{'east': 7, 'west': 2}`. **Constraints:** make the missing-key initial value explicit and preserve encounter order in the list version. **Verify:** assert all three outputs.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Given `[('east', 4), ('west', 2), ('east', 3)]`, build three mappings: region to list of all values, region to set of unique values, and region to numeric total. `{'east': 7, 'w...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict the length and membership behavior of `{3, 1, 3, 2}`. Why must display order not be treated as a sorting guarantee? **Progressive hint:** Sets enforce uniqueness and optimize membership, not presentation order. **Verify:** Assert the set has length `3` and contains `1`, `2`, and `3`; compare values as a set rather than asserting display iteration order.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict the length and membership behavior of `{3, 1, 3, 2}`. Why must display order not be treated as a sorting guarantee? Sets enforce uniqueness and optimize membership, not...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace shallow copying for `original = [[1], [2]]; copied = original.copy(); copied[0].append(9)`. **Progressive hint:** The outer lists differ but still refer to the same inner lists. **Verify:** Assert `original is not copied` but `original[0] is copied[0]`, then confirm appending through the copy appears in both nested lists.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace shallow copying for `original = [[1], [2]]; copied = original.copy(); copied[0].append(9)`. The outer lists differ but still refer to the same inner lists. Assert `origina...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement `invert_multimap(mapping)` so values become keys and each new key maps to a list of original keys in encounter order. **Progressive hint:** Use `setdefault` or `defaultdict(list)`. **Verify:** Assert exact output lists and encounter order for a mapping where two original keys share a value; also test an empty mapping.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Implement `invert_multimap(mapping)` so values become keys and each new key maps to a list of original keys in encounter order. Use `setdefault` or `defaultdict(list)`. Assert e...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair code that attempts to use a list as a dictionary key. Explain hashability and choose a tuple when the sequence is an identity. **Progressive hint:** Dictionary keys must have a stable hash while stored. **Verify:** Show the list-key version raises `TypeError`, then assert the tuple-key replacement retrieves the intended value and remains unmodified.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair code that attempts to use a list as a dictionary key. Explain hashability and choose a tuple when the sequence is an identity. Dictionary keys must have a stable hash whi...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Extend stable de-duplication to records using a `key` function, then test repeated dictionaries whose IDs match but other fields differ. **Progressive hint:** Store hashable derived keys while returning original records. **Verify:** Use repeated record dictionaries with the same ID; assert the first complete record is returned once, input order is preserved, and the input remains unchanged.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Extend stable de-duplication to records using a `key` function, then test repeated dictionaries whose IDs match but other fields differ. Store hashable derived keys while return...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
